# Phishing Website Detection: Reproducible Training and Evaluation

This notebook contains the **actual executable model training, cross-validation, hyperparameter tuning, holdout evaluation, and feature selection**. The archived `results/` files document an earlier experiment and are **not** automatically regenerated or asserted to match this notebook exactly.

**Protocol:** split once (80/20, stratified, seed 42); compare supervised models using 5-fold CV *on the training set*; tune the Random Forest on training folds only; evaluate the tuned model once on the held-out test set. Feature selection is determined exclusively from training data.

**Label convention:** `-1 = phishing` and `1 = legitimate`. Weighted F1 summarizes both classes; phishing recall specifically measures how many phishing sites are caught.

Run from the repository root or the `notebook/` folder after installing `requirements.txt`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score, make_scorer
)
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC

RANDOM_STATE = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = {
    "weighted_f1": "f1_weighted",
    "phishing_recall": make_scorer(recall_score, pos_label=-1),
}

## 1. Load and validate the dataset

Paths work when Jupyter starts in the repository root or its `notebook/` directory. No network download is needed.

In [ ]:
candidates = [
    Path.cwd() / "data" / "phishing.csv",
    Path.cwd().parent / "data" / "phishing.csv",
]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError("data/phishing.csv not found. Start Jupyter from the repository root or notebook/.")

df = pd.read_csv(data_path)
assert {"Index", "class"}.issubset(df.columns), "Expected Index and class columns"
assert set(df["class"].dropna().unique()) == {-1, 1}, "Unexpected target labels"
assert not df.isna().any().any(), "Missing values require a documented preprocessing step"
assert not df.drop(columns=["Index", "class"]).isna().any().any()

print("Shape:", df.shape)
print("Duplicate records (all columns):", int(df.duplicated().sum()))
print("Class distribution:\n", df["class"].value_counts().sort_index())

## 2. Engineer features without target leakage

The three scores are computed **only** from the 30 original predictor columns. Neither `class` nor `Index` participates.

In [ ]:
original_features = [c for c in df.columns if c not in ("Index", "class")]
features = df[original_features].copy()
features["Suspicion Score"] = (features[original_features] == -1).sum(axis=1)
features["Safe Score"] = (features[original_features] == 1).sum(axis=1)
features["Neutral Score"] = (features[original_features] == 0).sum(axis=1)
X = features
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Training rows: {len(X_train)} | Held-out test rows: {len(X_test)}")
assert X_train.index.intersection(X_test.index).empty

## 3. Compare supervised baselines using training-only cross-validation

Scaling is fitted **inside** each CV fold for algorithms that need it. The holdout test set is not used to select a model.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(random_state=RANDOM_STATE)),
    ]),
    "KNN": Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5)),
    ]),
}

baseline_rows = []
for name, estimator in models.items():
    scores = cross_validate(estimator, X_train, y_train, cv=CV, scoring=SCORING, n_jobs=-1)
    baseline_rows.append({
        "Model": name,
        "CV weighted F1 mean": scores["test_weighted_f1"].mean(),
        "CV weighted F1 std": scores["test_weighted_f1"].std(),
        "CV phishing recall mean": scores["test_phishing_recall"].mean(),
    })
baseline_results = pd.DataFrame(baseline_rows).sort_values("CV weighted F1 mean", ascending=False)
print(baseline_results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## 4. Random Forest hyperparameter search (training folds only)

This project investigates Random Forest as its primary detector; its final test score is reported even if another baseline ranks higher in the cross-validation table. This avoids silently changing the target model after seeing test results.

In [ ]:
param_distributions = {
    "n_estimators": [150, 250, 350],
    "max_depth": [None, 20, 30],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", None],
    "bootstrap": [True, False],
}
tuner = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    param_distributions=param_distributions,
    n_iter=20,
    scoring="f1_weighted",
    cv=CV,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
    error_score="raise",
)
tuner.fit(X_train, y_train)
best_rf = tuner.best_estimator_
print("Best CV weighted F1:", round(tuner.best_score_, 4))
print("Best hyperparameters:", tuner.best_params_)

## 5. Single holdout evaluation

The negative class (`-1`) is phishing. For confusion matrix ordered `[-1, 1]`, the top-right entry represents *missed phishing sites* (phishing predicted legitimate).

In [ ]:
def metrics_row(y_true, predicted):
    return {
        "accuracy": float(accuracy_score(y_true, predicted)),
        "weighted_f1": float(f1_score(y_true, predicted, average="weighted")),
        "phishing_precision": float(precision_score(y_true, predicted, pos_label=-1, zero_division=0)),
        "phishing_recall": float(recall_score(y_true, predicted, pos_label=-1, zero_division=0)),
    }

predictions = best_rf.predict(X_test)
test_metrics = metrics_row(y_test, predictions)
print("Held-out test performance:")
print(json.dumps(test_metrics, indent=2))
print("Confusion matrix: rows=actual, columns=predicted; labels=[-1 phishing, 1 legitimate]")
print(confusion_matrix(y_test, predictions, labels=[-1, 1]))

ConfusionMatrixDisplay.from_predictions(
    y_test, predictions, labels=[-1, 1],
    display_labels=["Phishing (-1)", "Legitimate (1)"],
    cmap="Blues", colorbar=False
)
plt.title("Tuned Random Forest: held-out test")
plt.tight_layout()
plt.show()

## 6. Feature selection without test leakage

The top 15 features are selected using the *trained model's feature importances from training data only*. The reduced model is fitted on training data and evaluated separately on the same held-out test data. Results are exploratory, not an additional independent confirmation.

In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
selected_features = importances.head(15).index.tolist()
reduced_rf = RandomForestClassifier(**best_rf.get_params())
reduced_rf.fit(X_train[selected_features], y_train)
reduced_predictions = reduced_rf.predict(X_test[selected_features])
reduced_metrics = metrics_row(y_test, reduced_predictions)
print("Top 15 features:", selected_features)
print("Reduced model holdout metrics:", json.dumps(reduced_metrics, indent=2))
importances.head(15).sort_values().plot.barh(figsize=(8, 6))
plt.title("Random Forest top 15 training-derived feature importances")
plt.xlabel("Feature importance")
plt.tight_layout()
plt.show()

## 7. Isolation Forest: separate unsupervised comparison

Isolation Forest is *not* a supervised classifier, and it is not treated as interchangeable with the supervised CV baselines. The contamination parameter below is an exploratory assumption informed by the **training labels**; no test labels enter the fit.

In [ ]:
unsupervised_model = Pipeline([
    ("scale", StandardScaler()),
    ("model", IsolationForest(
        contamination=float((y_train == -1).mean()),
        random_state=RANDOM_STATE,
        n_estimators=200,
        n_jobs=-1
    )),
])
unsupervised_model.fit(X_train)
unsupervised_predictions = unsupervised_model.predict(X_test)
print("Isolation Forest exploratory holdout metrics:")
print(json.dumps(metrics_row(y_test, unsupervised_predictions), indent=2))

## 8. Interpretation and limitations

- The archived `results/` metrics came from an earlier run, and should be considered **historical** until reproduced with the exact original setup.
- Running this notebook produces new outputs with fixed seeds but library versions and hardware can still affect results.
- Comparing algorithms using training-only CV is better than selecting based on test scores, but this single dataset still may not represent new phishing techniques.
- The three engineered scores summarize existing attributes; they do **not** represent newly collected independent signals.
- A real-time deployment requires a reliable feature extractor and evaluation on newer, out-of-distribution websites.
- Do not claim operational detection effectiveness based on offline test metrics alone.